In [ ]:
try:
    import dolfinx
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O "/tmp/fenicsx-install.sh" && bash "/tmp/fenicsx-install.sh"
    import dolfinx

In [ ]:
%cd /content
!rm -rf beam_fem_ml

!git clone https://github.com/adnan-math/beam_fem_ml.git
%cd beam_fem_ml

In [ ]:
!python data_generator.py

In [ ]:
!python training.py

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import torch.nn as nn

# ============================================================
# LOAD DATA
# ============================================================

data = np.load("beam_fem_dataset.npz")
stats = np.load("norm_stats.npz")

X = data["X"]
y = data["y"]

X_mean, X_std = stats["X_mean"], stats["X_std"]
y_mean, y_std = stats["y_mean"], stats["y_std"]

# ============================================================
# LOAD MODEL
# ============================================================

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)

model = MLP()
model.load_state_dict(torch.load("beam_mlp_model.pt"))
model.eval()

# ============================================================
# NORMALIZE INPUT
# ============================================================

Xn = (X - X_mean) / X_std

X_tensor = torch.tensor(Xn, dtype=torch.float32)

with torch.no_grad():
    pred_n = model(X_tensor).numpy().flatten()

pred = pred_n * y_std + y_mean

# ============================================================
# ERROR METRICS (INTERPOLATION / EXTRAPOLATION )
# ============================================================

L_values = np.unique(X[:, 0])

# MUST MATCH TRAINING SCRIPT
L_interp = np.array([0.8, 1.0, 1.2])

L_extrap = np.array([
    L for L in L_values
    if (L < 0.7) or (L > 1.3)
])

interp_mask = np.isin(X[:, 0], L_interp)
extrap_mask = np.isin(X[:, 0], L_extrap)

# ============================================================
# ERROR METRICS
# ============================================================

def compute_metrics(y_true, y_pred):

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    rel_l2 = np.linalg.norm(y_true - y_pred) / np.linalg.norm(y_true)

    return mse, rmse, mae, rel_l2


mse_i, rmse_i, mae_i, rel_i = compute_metrics(y[interp_mask],pred[interp_mask])
print("\nINTERPOLATION")
print("MSE :", mse_i)
print("RMSE:", rmse_i)
print("MAE :", mae_i)
print("Rel L2:", rel_i)


mse_e, rmse_e, mae_e, rel_e = compute_metrics(y[extrap_mask],pred[extrap_mask])
print("\nEXTRAPOLATION")
print("MSE :", mse_e)
print("RMSE:", rmse_e)
print("MAE :", mae_e)
print("Rel L2:", rel_e)



# ============================================================
# ERROR CURVES PER LENGTH
# ============================================================

for L in L_values:
    mask = np.isclose(X[:, 0], L)

    err = np.mean(np.abs(y[mask] - pred[mask]))
    print(f"L = {L:.3f} | MAE = {err:.6e}")

In [ ]:
# ============================================================
# FEM vs NN (INTERPOLATION - RUNTIME FEM GENERATION)
# ============================================================

from fem_dataset import BeamDataset
import numpy as np
import matplotlib.pyplot as plt
import torch

# create FEM generator object
dataset = BeamDataset()

# choose unseen interpolation value
L_plot = 0.79

# ============================================================
# GENERATE FEM SOLUTION ON THE FLY
# ============================================================

data_L = dataset.generate(L_plot, n_points=100)

x = data_L[:, 1]
y_true = data_L[:, 2]

# ============================================================
# NN PREDICTION
# ============================================================

L_array = np.full_like(x, L_plot)

X_nn = np.stack([L_array, x], axis=1)

# normalize
X_nn = (X_nn - X_mean) / X_std

X_tensor = torch.tensor(X_nn, dtype=torch.float32)

with torch.no_grad():
    y_pred_n = model(X_tensor).numpy().flatten()

# denormalize
y_pred = y_pred_n * y_std + y_mean

# ============================================================
# SORT FOR PLOTTING
# ============================================================

idx = np.argsort(x)
x = x[idx]
y_true = y_true[idx]
y_pred = y_pred[idx]

# ============================================================
# ERROR
# ============================================================

rel_l2 = np.linalg.norm(y_true - y_pred) / np.linalg.norm(y_true)

# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(8,5))
plt.plot(x, y_true, label="FEM (runtime)", linewidth=2)
plt.plot(x, y_pred, "--", label="NN surrogate", linewidth=2)

plt.xlabel("x")
plt.ylabel(r"$u_z$")
plt.title(
    f"Interpolation (Runtime FEM) "
    f"(L = {L_plot:.2f}, Rel. L2 = {100*rel_l2:.2f}%)"
)

plt.legend()
plt.grid()
plt.show()

In [ ]:
from fem_dataset import BeamDataset
import numpy as np
import matplotlib.pyplot as plt
import torch

dataset = BeamDataset()

# ============================================================
# EXTRAPOLATION RANGE
# ============================================================

L_extrap_curve = np.arange(0.5, 0.7, 0.02)

rel_l2_errors = []

# ============================================================
# LOOP OVER L VALUES
# ============================================================

for L_plot in L_extrap_curve:

    # -----------------------------
    # FEM solution (ground truth)
    # -----------------------------
    data_L = dataset.generate(L_plot, n_points=100)

    x = data_L[:, 1]
    y_true = data_L[:, 2]

    # -----------------------------
    # NN prediction
    # -----------------------------
    L_array = np.full_like(x, L_plot)

    X_nn = np.stack([L_array, x], axis=1)

    X_nn = (X_nn - X_mean) / X_std

    X_tensor = torch.tensor(X_nn, dtype=torch.float32)

    with torch.no_grad():
        y_pred_n = model(X_tensor).numpy().flatten()

    y_pred = y_pred_n * y_std + y_mean

    # -----------------------------
    # error
    # -----------------------------
    rel_l2 = np.linalg.norm(y_true - y_pred) / np.linalg.norm(y_true)

    rel_l2_errors.append(rel_l2)

# ============================================================
# PLOT ERROR CURVE
# ============================================================

plt.figure(figsize=(8,5))
plt.plot(L_extrap_curve, rel_l2_errors, marker='o', linewidth=2)

plt.xlabel("Beam length L")
plt.ylabel("Relative L2 error")
plt.title("Extrapolation Error Curve (L = 0.5 to 0.7)")
plt.grid()
plt.show()

In [ ]:
from fem_dataset import BeamDataset
import numpy as np
import matplotlib.pyplot as plt
import torch

dataset = BeamDataset()

# ============================================================
# L RANGES
# ============================================================

L_extrap_left  = np.arange(0.5, 0.7, 0.02)
L_interp       = np.arange(0.7, 1.3, 0.04)
L_extrap_right = np.arange(1.3, 1.5, 0.02)

# ============================================================
# STORAGE
# ============================================================

err_left = []
err_interp = []
err_right = []

# ============================================================
# ERROR FUNCTION
# ============================================================

def compute_rel_l2(L_plot):

    data_L = dataset.generate(L_plot, n_points=100)

    x = data_L[:, 1]
    y_true = data_L[:, 2]

    L_array = np.full_like(x, L_plot)
    X_nn = np.stack([L_array, x], axis=1)

    X_nn = (X_nn - X_mean) / X_std

    X_tensor = torch.tensor(X_nn, dtype=torch.float32)

    with torch.no_grad():
        y_pred_n = model(X_tensor).numpy().flatten()

    y_pred = y_pred_n * y_std + y_mean

    return np.linalg.norm(y_true - y_pred) / np.linalg.norm(y_true)

# ============================================================
# LEFT EXTRAPOLATION
# ============================================================

for L in L_extrap_left:
    err_left.append(compute_rel_l2(L))

# ============================================================
# INTERPOLATION
# ============================================================

for L in L_interp:
    err_interp.append(compute_rel_l2(L))

# ============================================================
# RIGHT EXTRAPOLATION
# ============================================================

for L in L_extrap_right:
    err_right.append(compute_rel_l2(L))

# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(9,5))

plt.plot(L_extrap_left, err_left, marker='o', label="Extrapolation (0.5–0.7)")
plt.plot(L_interp, err_interp, marker='o', label="Interpolation (0.7–1.3)")
plt.plot(L_extrap_right, err_right, marker='o', label="Extrapolation (1.3–1.5)")

plt.xlabel("Beam length L")
plt.ylabel("Relative L2 error")
plt.title("Error Profile: Interpolation vs Extrapolation")
plt.legend()
plt.grid()
plt.show()